# PokéAtlas — Indexación con Gemini Embeddings + ChromaDB

En este notebook vamos a construir la **base vectorial** de PokéAtlas.

Flujo:

```text
CSV limpios
   ↓
Documentos de texto
   ↓
Chunking
   ↓
Gemini Embeddings
   ↓
ChromaDB
   ↓
Búsqueda semántica
```

Este notebook **todavía no genera respuestas con el LLM**. Su objetivo es comprobar que PokéAtlas es capaz de recuperar correctamente información de nuestra base documental.

### Fuentes utilizadas

- `pokemon_complete_clean.csv`
- `type_relations.csv`
- `moves.csv`
- `abilities.csv`

La base vectorial se guardará en `chroma_db/` para poder reutilizarla posteriormente desde el agente LangGraph.


## 0. Dependencias

Las dependencias ya aparecen en `requirements.txt`. Si hiciera falta instalarlas manualmente:

```bash
pip install -U pandas python-dotenv langchain-core langchain-text-splitters langchain-google-genai langchain-chroma chromadb
```

Si actualizas paquetes dentro de Jupyter y aparece algún error extraño de importación, reinicia el kernel antes de continuar.


In [1]:
from pathlib import Path
import os
import shutil
import pandas as pd

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma


## 1. Localizar el proyecto y cargar la API key

El notebook funciona tanto si Jupyter toma como carpeta actual `notebooks/` como si toma la raíz de `PokeAtlas/`.


In [2]:
CWD = Path.cwd()

if (CWD / "data").exists() and (CWD / ".env").exists():
    ROOT = CWD
elif (CWD.parent / "data").exists() and (CWD.parent / ".env").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "No encuentro la raíz de PokeAtlas. "
        "Comprueba que existen las carpetas data/ y el archivo .env."
    )

DATA_DIR = ROOT / "data" / "knowledge"
CHROMA_DIR = ROOT / "chroma_db"

load_dotenv(ROOT / ".env")

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("No se ha encontrado GEMINI_API_KEY en el archivo .env")

print("✅ Raíz del proyecto:", ROOT)
print("✅ GEMINI_API_KEY encontrada.")
print("✅ Carpeta de conocimiento:", DATA_DIR)
print("✅ ChromaDB se guardará en:", CHROMA_DIR)


✅ Raíz del proyecto: c:\Users\alvar\OneDrive\Desktop\Ejercicio asistente gemini evolve academy\PokeAtlas
✅ GEMINI_API_KEY encontrada.
✅ Carpeta de conocimiento: c:\Users\alvar\OneDrive\Desktop\Ejercicio asistente gemini evolve academy\PokeAtlas\data\knowledge
✅ ChromaDB se guardará en: c:\Users\alvar\OneDrive\Desktop\Ejercicio asistente gemini evolve academy\PokeAtlas\chroma_db


## 2. Cargar las cuatro fuentes

Antes de generar embeddings verificamos que los cuatro CSV existen y podemos leerlos correctamente.


In [3]:
paths = {
    "pokemon": DATA_DIR / "pokemon_complete_clean.csv",
    "types": DATA_DIR / "type_relations.csv",
    "moves": DATA_DIR / "moves.csv",
    "abilities": DATA_DIR / "abilities.csv",
}

missing = [str(path) for path in paths.values() if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Faltan archivos de conocimiento:\n- " + "\n- ".join(missing)
    )

pokemon_df = pd.read_csv(paths["pokemon"])
types_df = pd.read_csv(paths["types"])
moves_df = pd.read_csv(paths["moves"])
abilities_df = pd.read_csv(paths["abilities"])

print("FUENTES CARGADAS")
print("=" * 55)
print("Pokémon:    ", pokemon_df.shape)
print("Tipos:      ", types_df.shape)
print("Movimientos:", moves_df.shape)
print("Habilidades:", abilities_df.shape)


FUENTES CARGADAS
Pokémon:     (1350, 31)
Tipos:       (18, 7)
Movimientos: (937, 11)
Habilidades: (310, 6)


## 3. Preparar funciones auxiliares

Los embeddings funcionan mejor cuando reciben **texto con significado**, no simplemente una fila de números.

Por eso convertiremos cada registro en una pequeña ficha textual. Además, eliminamos valores `NaN` y utilizamos metadatos para saber posteriormente de qué fuente procede cada fragmento.


In [4]:
TYPE_ES = {
    "normal": "normal",
    "fire": "fuego",
    "water": "agua",
    "electric": "eléctrico",
    "grass": "planta",
    "ice": "hielo",
    "fighting": "lucha",
    "poison": "veneno",
    "ground": "tierra",
    "flying": "volador",
    "psychic": "psíquico",
    "bug": "bicho",
    "rock": "roca",
    "ghost": "fantasma",
    "dragon": "dragón",
    "dark": "siniestro",
    "steel": "acero",
    "fairy": "hada",
}

DAMAGE_CLASS_ES = {
    "physical": "físico",
    "special": "especial",
    "status": "estado",
}

def clean_value(value, default="no consta"):
    if pd.isna(value):
        return default
    text = str(value).strip()
    return text if text else default


def pretty_identifier(value):
    if pd.isna(value):
        return "no consta"
    return str(value).replace("-", " ").strip()


def translate_type(value):
    if pd.isna(value):
        return "no consta"
    value = str(value).strip().lower()
    return f"{TYPE_ES.get(value, value)} ({value})"


def pipe_to_spanish(value):
    if pd.isna(value) or not str(value).strip():
        return "ninguno"

    values = [x.strip().lower() for x in str(value).split("|") if x.strip()]
    return ", ".join(
        f"{TYPE_ES.get(x, x)} ({x})"
        for x in values
    )


## 4. Convertir Pokémon a documentos

Cada fila representa una especie o forma y funciona como una **unidad semántica natural**.

También corregimos un detalle del dataset: la columna `abilities` puede incluir la habilidad oculta. La separamos para no presentarla como si fuera una habilidad estándar.


In [5]:
pokemon_docs = []

for idx, row in pokemon_df.iterrows():
    name = clean_value(row.get("name"))
    hidden = clean_value(row.get("hidden_ability"), default="").lower()

    all_abilities = [
        a.strip()
        for a in str(row.get("abilities", "")).split("|")
        if a.strip() and a.strip().lower() != "nan"
    ]

    standard_abilities = [
        a for a in all_abilities
        if not hidden or a.lower() != hidden
    ]

    standard_text = (
        ", ".join(pretty_identifier(a) for a in standard_abilities)
        if standard_abilities
        else "no consta"
    )

    hidden_text = pretty_identifier(hidden) if hidden else "no consta"

    text = f"""
Pokémon: {name}.
Número de Pokédex: {clean_value(row.get('pokedex_number'))}.
Generación: {clean_value(row.get('generation'))}.
Tipo principal: {translate_type(row.get('type_1'))}.
Tipo secundario: {translate_type(row.get('type_2'))}.
Estadísticas base: PS {clean_value(row.get('hp'))}, Ataque {clean_value(row.get('attack'))},
Defensa {clean_value(row.get('defense'))}, Ataque Especial {clean_value(row.get('sp_attack'))},
Defensa Especial {clean_value(row.get('sp_defense'))}, Velocidad {clean_value(row.get('speed'))}.
Total de estadísticas base: {clean_value(row.get('base_stat_total'))}.
Altura: {clean_value(row.get('height_m'))} metros.
Peso: {clean_value(row.get('weight_kg'))} kg.
Habilidades estándar: {standard_text}.
Habilidad oculta: {hidden_text}.
Legendario: {clean_value(row.get('is_legendary'))}.
Mítico: {clean_value(row.get('is_mythical'))}.
Bebé: {clean_value(row.get('is_baby'))}.
Género o categoría Pokédex: {clean_value(row.get('genus'))}.
Grupo huevo: {clean_value(row.get('egg_groups'))}.
Hábitat: {clean_value(row.get('habitat'))}.
Ritmo de crecimiento: {clean_value(row.get('growth_rate'))}.
Ratio de captura: {clean_value(row.get('capture_rate'))}.
Descripción Pokédex: {clean_value(row.get('flavor_text'))}.
""".strip()

    pokemon_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "pokemon_complete_clean.csv",
                "category": "pokemon",
                "name": str(name),
                "row": int(idx),
            },
        )
    )

print("Documentos Pokémon creados:", len(pokemon_docs))
print()
print(pokemon_docs[0].page_content[:1000])


Documentos Pokémon creados: 1350

Pokémon: Bulbasaur.
Número de Pokédex: 1.
Generación: gen-i.
Tipo principal: planta (grass).
Tipo secundario: veneno (poison).
Estadísticas base: PS 45, Ataque 49,
Defensa 49, Ataque Especial 65,
Defensa Especial 65, Velocidad 45.
Total de estadísticas base: 318.
Altura: 0.7 metros.
Peso: 6.9 kg.
Habilidades estándar: overgrow.
Habilidad oculta: chlorophyll.
Legendario: False.
Mítico: False.
Bebé: False.
Género o categoría Pokédex: Seed Pokémon.
Grupo huevo: monster|plant.
Hábitat: grassland.
Ritmo de crecimiento: medium-slow.
Ratio de captura: 45.
Descripción Pokédex: A strange seed was planted on its back at birth. The plant sprouts and grows with this POKéMON..


## 5. Convertir relaciones de tipos a documentos

Cada tipo tendrá una ficha con sus relaciones ofensivas y defensivas.


In [6]:
type_docs = []

for idx, row in types_df.iterrows():
    type_en = clean_value(row.get("type"))
    type_es = TYPE_ES.get(str(type_en).lower(), str(type_en))

    text = f"""
Tipo Pokémon: {type_es} ({type_en}).

Relaciones ofensivas:
- Hace daño doble (x2) a: {pipe_to_spanish(row.get('double_damage_to'))}.
- Hace medio daño (x0.5) a: {pipe_to_spanish(row.get('half_damage_to'))}.
- No hace daño (x0) a: {pipe_to_spanish(row.get('no_damage_to'))}.

Relaciones defensivas:
- Recibe daño doble (x2) de: {pipe_to_spanish(row.get('double_damage_from'))}.
- Recibe medio daño (x0.5) de: {pipe_to_spanish(row.get('half_damage_from'))}.
- Es inmune al daño de: {pipe_to_spanish(row.get('no_damage_from'))}.
""".strip()

    type_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "type_relations.csv",
                "category": "type",
                "name": str(type_en),
                "row": int(idx),
            },
        )
    )

print("Documentos de tipos creados:", len(type_docs))
print()
print(type_docs[0].page_content)


Documentos de tipos creados: 18

Tipo Pokémon: bicho (bug).

Relaciones ofensivas:
- Hace daño doble (x2) a: siniestro (dark), planta (grass), psíquico (psychic).
- Hace medio daño (x0.5) a: hada (fairy), lucha (fighting), fuego (fire), volador (flying), fantasma (ghost), veneno (poison), acero (steel).
- No hace daño (x0) a: ninguno.

Relaciones defensivas:
- Recibe daño doble (x2) de: fuego (fire), volador (flying), roca (rock).
- Recibe medio daño (x0.5) de: lucha (fighting), planta (grass), tierra (ground).
- Es inmune al daño de: ninguno.


## 6. Convertir movimientos a documentos

Los nombres originales de PokéAPI se conservan como identificadores. Gemini Embeddings es multilingüe, por lo que una consulta en español puede recuperar contenido relacionado aunque parte del documento esté en inglés.


In [7]:
move_docs = []

for idx, row in moves_df.iterrows():
    move = clean_value(row.get("move"))
    damage_class = clean_value(row.get("damage_class"))
    damage_class_es = DAMAGE_CLASS_ES.get(str(damage_class).lower(), damage_class)

    text = f"""
Movimiento Pokémon: {pretty_identifier(move)}.
Identificador original: {move}.
Generación: {clean_value(row.get('generation'))}.
Tipo: {translate_type(row.get('type'))}.
Categoría: {damage_class_es} ({damage_class}).
Potencia: {clean_value(row.get('power'))}.
Precisión: {clean_value(row.get('accuracy'))}.
PP: {clean_value(row.get('pp'))}.
Prioridad: {clean_value(row.get('priority'))}.
Probabilidad de efecto secundario: {clean_value(row.get('effect_chance'))}.
Efecto: {clean_value(row.get('short_effect'))}.
""".strip()

    move_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "moves.csv",
                "category": "move",
                "name": str(move),
                "row": int(idx),
            },
        )
    )

print("Documentos de movimientos creados:", len(move_docs))
print()
print(move_docs[0].page_content)


Documentos de movimientos creados: 937

Movimiento Pokémon: pound.
Identificador original: pound.
Generación: gen-i.
Tipo: normal (normal).
Categoría: físico (physical).
Potencia: 40.0.
Precisión: 100.0.
PP: 35.0.
Prioridad: 0.
Probabilidad de efecto secundario: no consta.
Efecto: Inflicts regular damage with no additional effect..


## 7. Convertir habilidades a documentos

Conservamos tanto el efecto corto como la explicación ampliada de PokéAPI.


In [8]:
ability_docs = []

for idx, row in abilities_df.iterrows():
    ability = clean_value(row.get("ability"))

    text = f"""
Habilidad Pokémon: {pretty_identifier(ability)}.
Identificador original: {ability}.
Generación: {clean_value(row.get('generation'))}.
Pertenece a la serie principal: {clean_value(row.get('is_main_series'))}.
Resumen del efecto: {clean_value(row.get('short_effect'))}.
Descripción ampliada: {clean_value(row.get('effect'))}.
""".strip()

    ability_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": "abilities.csv",
                "category": "ability",
                "name": str(ability),
                "row": int(idx),
            },
        )
    )

print("Documentos de habilidades creados:", len(ability_docs))
print()
print(ability_docs[0].page_content[:1000])


Documentos de habilidades creados: 310

Habilidad Pokémon: stench.
Identificador original: stench.
Generación: gen-iii.
Pertenece a la serie principal: 1.
Resumen del efecto: Has a 10% chance of making target Pokémon flinch with each hit..
Descripción ampliada: This Pokémon's damaging moves have a 10% chance to make the target flinch with each hit if they do not already cause flinching as a secondary effect.

This ability does not stack with a held item.

Overworld: The wild encounter rate is halved while this Pokémon is first in the party..


## 8. Unir documentos y aplicar chunking

Cada fila sigue representando una unidad lógica, pero el nivel gratuito de Gemini aplica un límite de solicitudes de embeddings por minuto.

Para mantener el proyecto dentro del nivel gratuito, agrupamos varias fichas completas de la **misma categoría** en chunks de hasta aproximadamente 4.500 caracteres. Así:

- no mezclamos Pokémon con movimientos, tipos o habilidades;
- no cortamos una ficha por la mitad salvo que sea excepcionalmente larga;
- reducimos mucho el número de embeddings;
- mantenemos suficiente precisión para la recuperación semántica.

La decisión queda además documentada como parte del diseño del RAG.


In [9]:
# ------------------------------------------------------------------
# Chunking adaptado al límite gratuito de Gemini Embeddings
# ------------------------------------------------------------------
# Cada fila sigue siendo una unidad semántica, pero agrupamos varias fichas
# completas en un mismo chunk. Así reducimos mucho el número de embeddings
# sin cortar entidades por la mitad.

def pack_documents(docs, max_chars=4500):
    packed = []
    current = []
    current_chars = 0

    def flush(group):
        if not group:
            return

        content = "\n\n---\n\n".join(doc.page_content for doc in group)
        names = " | ".join(str(doc.metadata.get("name", "")) for doc in group)

        packed.append(
            Document(
                page_content=content,
                metadata={
                    "source": group[0].metadata.get("source", ""),
                    "category": group[0].metadata.get("category", ""),
                    "names": names,
                    "records": len(group),
                },
            )
        )

    for doc in docs:
        text_len = len(doc.page_content)

        # Si una ficha individual fuese excepcionalmente larga,
        # la dividimos de forma segura antes de empaquetarla.
        if text_len > max_chars:
            flush(current)
            current = []
            current_chars = 0

            long_splitter = RecursiveCharacterTextSplitter(
                chunk_size=max_chars,
                chunk_overlap=200,
                separators=["\n\n", "\n", ". ", " ", ""],
            )

            for piece in long_splitter.split_documents([doc]):
                piece.metadata["names"] = str(doc.metadata.get("name", ""))
                piece.metadata["records"] = 1
                packed.append(piece)

            continue

        extra = text_len + (7 if current else 0)  # separador aproximado

        if current and current_chars + extra > max_chars:
            flush(current)
            current = []
            current_chars = 0

        current.append(doc)
        current_chars += extra

    flush(current)
    return packed


# Empaquetamos por categoría para no mezclar Pokémon, tipos,
# movimientos y habilidades dentro del mismo embedding.
chunks = []
chunks.extend(pack_documents(pokemon_docs, max_chars=4500))
chunks.extend(pack_documents(type_docs, max_chars=4500))
chunks.extend(pack_documents(move_docs, max_chars=4500))
chunks.extend(pack_documents(ability_docs, max_chars=4500))

print("Documentos lógicos originales:", len(
    pokemon_docs + type_docs + move_docs + ability_docs
))
print("Chunks optimizados para embeddings:", len(chunks))

counts = {}
for chunk in chunks:
    category = chunk.metadata["category"]
    counts[category] = counts.get(category, 0) + 1

print("\nChunks por categoría:")
for category, count in counts.items():
    print(f"- {category}: {count}")

print("\nEjemplo de chunk:")
print("=" * 80)
print(chunks[0].page_content[:1500])


Documentos lógicos originales: 2615
Chunks optimizados para embeddings: 333

Chunks por categoría:
- pokemon: 229
- type: 2
- move: 68
- ability: 34

Ejemplo de chunk:
Pokémon: Bulbasaur.
Número de Pokédex: 1.
Generación: gen-i.
Tipo principal: planta (grass).
Tipo secundario: veneno (poison).
Estadísticas base: PS 45, Ataque 49,
Defensa 49, Ataque Especial 65,
Defensa Especial 65, Velocidad 45.
Total de estadísticas base: 318.
Altura: 0.7 metros.
Peso: 6.9 kg.
Habilidades estándar: overgrow.
Habilidad oculta: chlorophyll.
Legendario: False.
Mítico: False.
Bebé: False.
Género o categoría Pokédex: Seed Pokémon.
Grupo huevo: monster|plant.
Hábitat: grassland.
Ritmo de crecimiento: medium-slow.
Ratio de captura: 45.
Descripción Pokédex: A strange seed was planted on its back at birth. The plant sprouts and grows with this POKéMON..

---

Pokémon: Ivysaur.
Número de Pokédex: 2.
Generación: gen-i.
Tipo principal: planta (grass).
Tipo secundario: veneno (poison).
Estadísticas base: PS 60, At

## 9. Crear Gemini Embeddings

Utilizamos `gemini-embedding-001`.

No fijamos manualmente `task_type`: la integración actual de LangChain utiliza el tipo adecuado para documentos al indexar y para consultas al buscar.


In [10]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

# Prueba pequeña antes de indexar miles de documentos
test_vector = embeddings.embed_query(
    "¿Qué estadísticas base tiene Garchomp?"
)

print("✅ Gemini Embeddings funciona.")
print("Dimensiones del vector:", len(test_vector))


✅ Gemini Embeddings funciona.
Dimensiones del vector: 3072


### Nota sobre el nivel gratuito

La prueba anterior consume una cantidad mínima de cuota. La indexación completa se realiza después con tandas pequeñas y pausas automáticas para respetar tanto el número de solicitudes como el presupuesto de tokens por minuto.


## 10. Crear ChromaDB respetando las cuotas gratuitas

⚠️ **Esta celda reinicia la colección vectorial antes de indexar.**

En Windows seguimos utilizando `reset_collection()` para no borrar físicamente el archivo SQLite.

Además, el error real de Gemini nos ha mostrado dos restricciones diferentes del nivel gratuito:

- límite de solicitudes;
- límite de **tokens de embeddings por minuto**.

Por eso usamos tandas pequeñas de **15 chunks** y dejamos pasar una ventana completa entre tandas. Si Gemini devuelve un `429`, el código lee el tiempo de reintento cuando aparece en el mensaje y vuelve a intentar automáticamente la misma tanda.

Esta estrategia es más conservadora, pero evita superar la cuota sin activar facturación.


In [11]:
import time
import re

REBUILD_DB = True

vectorstore = Chroma(
    collection_name="pokeatlas_knowledge",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

if REBUILD_DB:
    print("Reiniciando la colección Chroma sin borrar el archivo SQLite...")
    vectorstore.reset_collection()
    print("✅ Colección vaciada y recreada correctamente.")

# ------------------------------------------------------------------
# Control de cuota del nivel gratuito de Gemini Embeddings
# ------------------------------------------------------------------
# El error real observado indica un límite de 30.000 tokens/minuto.
# 75 chunks de hasta ~4.500 caracteres pueden superar fácilmente ese
# presupuesto. Bajamos a tandas mucho más pequeñas y dejamos margen.
BATCH_SIZE = 15
DEFAULT_PAUSE_SECONDS = 65
MAX_RETRIES = 5

total = len(chunks)

def retry_delay_from_error(message, default=65):
    """
    Intenta leer mensajes como:
    'Please retry in 58.52s'
    Si no aparecen, devuelve una pausa conservadora.
    """
    match = re.search(r"retry in\s+([0-9.]+)s", message, flags=re.IGNORECASE)
    if match:
        return max(float(match.group(1)) + 2, default)
    return default


for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    batch = chunks[start:end]

    print(f"\nIndexando chunks {start + 1}-{end} de {total}...")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vectorstore.add_documents(batch)

            print(
                f"✅ Tanda completada. "
                f"Vectores almacenados: {vectorstore._collection.count()}"
            )
            break

        except Exception as exc:
            message = str(exc)

            if "429" in message or "RESOURCE_EXHAUSTED" in message:
                if attempt == MAX_RETRIES:
                    raise

                pause = retry_delay_from_error(
                    message,
                    default=DEFAULT_PAUSE_SECONDS
                )

                print(
                    f"⚠️ Cuota temporal de Gemini alcanzada "
                    f"(intento {attempt}/{MAX_RETRIES})."
                )
                print(
                    f"Pausa automática de {pause:.0f} s "
                    "antes de reintentar esta misma tanda..."
                )
                time.sleep(pause)
            else:
                raise

    # Dejamos pasar una ventana completa antes de la siguiente tanda.
    if end < total:
        print(
            "Pausa automática entre tandas para respetar "
            "el límite de tokens/minuto..."
        )
        time.sleep(DEFAULT_PAUSE_SECONDS)

print("\n" + "=" * 70)
print("✅ Base vectorial creada correctamente.")
print("Colección: pokeatlas_knowledge")
print("Directorio:", CHROMA_DIR)
print("Vectores almacenados:", vectorstore._collection.count())


Reiniciando la colección Chroma sin borrar el archivo SQLite...
✅ Colección vaciada y recreada correctamente.

Indexando chunks 1-15 de 333...
✅ Tanda completada. Vectores almacenados: 15
Pausa automática entre tandas para respetar el límite de tokens/minuto...

Indexando chunks 16-30 de 333...
✅ Tanda completada. Vectores almacenados: 30
Pausa automática entre tandas para respetar el límite de tokens/minuto...

Indexando chunks 31-45 de 333...
✅ Tanda completada. Vectores almacenados: 45
Pausa automática entre tandas para respetar el límite de tokens/minuto...

Indexando chunks 46-60 de 333...
✅ Tanda completada. Vectores almacenados: 60
Pausa automática entre tandas para respetar el límite de tokens/minuto...

Indexando chunks 61-75 de 333...
✅ Tanda completada. Vectores almacenados: 75
Pausa automática entre tandas para respetar el límite de tokens/minuto...

Indexando chunks 76-90 de 333...
✅ Tanda completada. Vectores almacenados: 90
Pausa automática entre tandas para respetar el 

## 11. Primera búsqueda semántica

Todavía no interviene Gemini como LLM. Aquí solo hacemos:

```text
pregunta → embedding → ChromaDB → fragmentos más parecidos
```

Esto nos permite comprobar si el **retriever** encuentra información útil antes de construir el RAG completo.


In [12]:
def buscar(query, k=4):
    resultados = vectorstore.similarity_search(query, k=k)

    print(f"Pregunta: {query}")
    print("=" * 80)

    for i, doc in enumerate(resultados, start=1):
        print(
            f"\nRESULTADO {i} | "
            f"fuente={doc.metadata.get('source')} | "
            f"categoría={doc.metadata.get('category')} | "
            f"nombre={doc.metadata.get('name')}"
        )
        print("-" * 80)
        print(doc.page_content[:1200])

    return resultados


In [13]:
resultados_garchomp = buscar(
    "¿Qué tipos tiene Garchomp y cuáles son sus estadísticas base?",
    k=4
)


Pregunta: ¿Qué tipos tiene Garchomp y cuáles son sus estadísticas base?

RESULTADO 1 | fuente=pokemon_complete_clean.csv | categoría=pokemon | nombre=None
--------------------------------------------------------------------------------
Pokémon: Gabite.
Número de Pokédex: 444.
Generación: gen-iv.
Tipo principal: dragón (dragon).
Tipo secundario: tierra (ground).
Estadísticas base: PS 68, Ataque 90,
Defensa 65, Ataque Especial 50,
Defensa Especial 55, Velocidad 82.
Total de estadísticas base: 410.
Altura: 1.4 metros.
Peso: 56.0 kg.
Habilidades estándar: sand veil.
Habilidad oculta: rough skin.
Legendario: False.
Mítico: False.
Bebé: False.
Género o categoría Pokédex: Cave Pokémon.
Grupo huevo: monster|dragon.
Hábitat: no consta.
Ritmo de crecimiento: slow.
Ratio de captura: 45.
Descripción Pokédex: There is a long-held belief that medicine made from its scales will heal even incurable illnesses..

---

Pokémon: Garchomp.
Número de Pokédex: 445.
Generación: gen-iv.
Tipo principal: dragón 

In [14]:
resultados_movimiento = buscar(
    "¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?",
    k=4
)


Pregunta: ¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?

RESULTADO 1 | fuente=moves.csv | categoría=move | nombre=None
--------------------------------------------------------------------------------
Movimiento Pokémon: stone axe.
Identificador original: stone-axe.
Generación: gen-viii.
Tipo: roca (rock).
Categoría: físico (physical).
Potencia: 65.0.
Precisión: 90.0.
PP: 15.0.
Prioridad: 0.
Probabilidad de efecto secundario: no consta.
Efecto: no consta.

---

Movimiento Pokémon: springtide storm.
Identificador original: springtide-storm.
Generación: gen-viii.
Tipo: hada (fairy).
Categoría: especial (special).
Potencia: 100.0.
Precisión: 80.0.
PP: 5.0.
Prioridad: 0.
Probabilidad de efecto secundario: no consta.
Efecto: no consta.

---

Movimiento Pokémon: mystical power.
Identificador original: mystical-power.
Generación: gen-viii.
Tipo: psíquico (psychic).
Categoría: especial (special).
Potencia: 70.0.
Precisión: 90.0.
PP: 10.0.
Prioridad: 0.
Probabilidad de efecto sec

In [15]:
resultados_habilidad = buscar(
    "¿Qué efecto tiene la habilidad Rough Skin?",
    k=4
)


Pregunta: ¿Qué efecto tiene la habilidad Rough Skin?

RESULTADO 1 | fuente=abilities.csv | categoría=ability | nombre=None
--------------------------------------------------------------------------------
Habilidad Pokémon: flash fire.
Identificador original: flash-fire.
Generación: gen-iii.
Pertenece a la serie principal: 1.
Resumen del efecto: Protects against Fire moves.  Once one has been blocked, the Pokémon's own Fire moves inflict 1.5× damage until it leaves battle..
Descripción ampliada: This Pokémon is immune to Fire-type moves.  Once this Pokémon has been hit by a Fire move, its own Fire moves will inflict 1.5× as much damage until it leaves battle.

This ability has no effect while the Pokémon is frozen.  The Fire damage bonus is retained even if the Pokémon is frozen and thawed or the ability is lost or disabled.  Fire moves will ignore this Pokémon's Substitute.  This ability takes effect even on non-damaging moves, i.e. Will-O-Wisp..

---

Habilidad Pokémon: shield dust.
I

In [16]:
resultados_tipo = buscar(
    "¿De qué tipos recibe daño doble el tipo dragón?",
    k=4
)


Pregunta: ¿De qué tipos recibe daño doble el tipo dragón?

RESULTADO 1 | fuente=type_relations.csv | categoría=type | nombre=None
--------------------------------------------------------------------------------
Tipo Pokémon: bicho (bug).

Relaciones ofensivas:
- Hace daño doble (x2) a: siniestro (dark), planta (grass), psíquico (psychic).
- Hace medio daño (x0.5) a: hada (fairy), lucha (fighting), fuego (fire), volador (flying), fantasma (ghost), veneno (poison), acero (steel).
- No hace daño (x0) a: ninguno.

Relaciones defensivas:
- Recibe daño doble (x2) de: fuego (fire), volador (flying), roca (rock).
- Recibe medio daño (x0.5) de: lucha (fighting), planta (grass), tierra (ground).
- Es inmune al daño de: ninguno.

---

Tipo Pokémon: siniestro (dark).

Relaciones ofensivas:
- Hace daño doble (x2) a: fantasma (ghost), psíquico (psychic).
- Hace medio daño (x0.5) a: siniestro (dark), hada (fairy), lucha (fighting).
- No hace daño (x0) a: ninguno.

Relaciones defensivas:
- Recibe daño

## 12. Reabrir una ChromaDB ya creada

En los siguientes notebooks **no volveremos a generar todos los embeddings**.

Bastará con cargar la misma colección desde disco:

```python
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore = Chroma(
    collection_name="pokeatlas_knowledge",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)
```

Así reutilizamos la base vectorial ya construida.


## Qué hemos conseguido

Al finalizar correctamente este notebook tendremos:

- cuatro fuentes documentales transformadas a texto;
- preprocesado y chunking;
- embeddings generados con Google Gemini;
- almacenamiento persistente en ChromaDB;
- recuperación semántica probada con varias consultas.

El siguiente paso será construir el **RAG**:

```text
Pregunta
   ↓
ChromaDB recupera contexto
   ↓
System Prompt + contexto
   ↓
Gemini genera una respuesta fundamentada
   ↓
LangGraph orquesta el flujo
   ↓
Memoria conversacional
```
